# 02 — Data Preprocessing & Pipeline Engineering
## Compact Encoding, Imputation, & Feature Scaling

### Key Architectural Learnings Implemented:
1. **Elimination of High-Dimensional One-Hot Sparsity:** Instead of one-hot encoding which creates 130+ sparse binary columns for proto, we implement compact OrdinalEncoder with unknown value clamping. This keeps tree split quality dense and sharp.
2. **StandardScaler Normalization:** Normalizing numerical flow metrics ($\\mu=0, \\sigma=1$) to prevent high-magnitude packet counters from dominating.
3. **Full Pipeline Packaging:** Combining steps into a reusable Scikit-Learn ColumnTransformer.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

DATA_DIR = Path('../data/raw')
train_df = pd.read_csv(DATA_DIR / 'UNSW_NB15_training-set.csv', low_memory=False)
test_df = pd.read_csv(DATA_DIR / 'UNSW_NB15_testing-set.csv', low_memory=False)

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

excluded = ['id', 'label', 'attack_cat']
X_train = train_df.drop(columns=excluded)
y_train = train_df['label'].astype(int)

X_test = test_df.drop(columns=excluded)
y_test = test_df['label'].astype(int)

categorical_columns = ['proto', 'service', 'state']
numerical_columns = [c for c in X_train.columns if c not in categorical_columns]

print(f'Input features: {X_train.shape[1]} (3 categorical, {len(numerical_columns)} numerical)')
print(f'Training rows:   {len(X_train)}')
print(f'Testing rows:    {len(X_test)}')


Input features: 42 (3 categorical, 39 numerical)
Training rows:   175341
Testing rows:    82332


### 1. Build Preprocessor
Using SimpleImputer for numerical robustness and compact OrdinalEncoder for categoricals.

In [2]:
def build_compact_preprocessor(numerical_cols, categorical_cols):
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    
    return ColumnTransformer([
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols)
    ])

preprocessor = build_compact_preprocessor(numerical_columns, categorical_columns)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f'Processed feature matrix shape: {X_train_proc.shape}')
print(f'Any NaN in processed train: {np.isnan(X_train_proc).any()}')
print(f'Any NaN in processed test:  {np.isnan(X_test_proc).any()}')


Processed feature matrix shape: (175341, 42)
Any NaN in processed train: False
Any NaN in processed test:  False


### 2. Verify Preprocessor Output
Compare raw distribution vs preprocessed standardized ranges.

In [3]:
proc_df = pd.DataFrame(X_train_proc[:, :5], columns=numerical_columns[:5])
print('First 5 preprocessed numerical features sample (mean ~0, std ~1):')
display(proc_df.describe().round(3))


First 5 preprocessed numerical features sample (mean ~0, std ~1):


,dur,spkts,dpkts,sbytes,dbytes
count,175341.000,175341.000,175341.000,175341.000,175341.000
mean,-0.000,-0.000,0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000
min,-0.210,-0.141,-0.172,-0.050,-0.104
25%,-0.210,-0.134,-0.172,-0.050,-0.104
50%,-0.210,-0.134,-0.154,-0.048,-0.103
75%,-0.107,-0.061,-0.081,-0.042,-0.096
max,9.049,70.099,99.358,74.136,101.916
